# ORI 2026 - KOMPLETAN NOTEBOOK V3
## SA TA PAKETOM ZA INDIKATORE + MISSING VALUE REPORT

Ovaj notebook ima:
- ✅ Učitavanje CSV fajlova (Date, Price, Open, High, Low, Vol.)
- ✅ Izveštaj o udelu nedostajućih vrednosti po valutnom paru
- ✅ Inženjering indikatora KORIŠĆENJEM GOTOVIH FUNKCIJA (paket `ta`)
- ✅ Neuronska mreža (MLP regresija + klasifikacija)

In [33]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
from math import sqrt

warnings.filterwarnings('ignore')

# Auto-pronalaženje CSV foldera
possible_dirs = [".", "data", "./data", "../data"]
INPUT_DIR = ".."

for dir_path in possible_dirs:
    if os.path.exists(dir_path):
        csv_files = glob.glob(os.path.join(dir_path, "*_Data.csv"))
        if csv_files and len(csv_files) >= 6:
            INPUT_DIR = dir_path
            break

print(f"✓ CSV fajlovi pronađeni u: {INPUT_DIR}")
print(f"✓ Sve biblioteke učitane!")

✓ CSV fajlovi pronađeni u: data
✓ Sve biblioteke učitane!


## 📋 FAZA 1: Priprema podataka

In [34]:
# Učitaj CSV fajlove (Date, Price, Open, High, Low, Vol., Change %)
def load_csv(path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [c.strip().strip('"') for c in df.columns]
    return df

def clean_numeric_column(series):
    """Konvertuje string kolonu (sa zarezima/procentima/K,M,B sufiksima) u numeričku."""
    s = series.astype(str).str.replace(",", "", regex=False)
    s = s.str.replace("%", "", regex=False)

    def parse_suffix(val):
        if not isinstance(val, str):
            return np.nan if pd.isna(val) else float(val)
        val = val.strip()
        if val in ("", "nan", "None", "-"):
            return np.nan
        multiplier = 1
        if val.endswith("K"):
            multiplier, val = 1_000, val[:-1]
        elif val.endswith("M"):
            multiplier, val = 1_000_000, val[:-1]
        elif val.endswith("B"):
            multiplier, val = 1_000_000_000, val[:-1]
        try:
            return float(val) * multiplier
        except ValueError:
            return np.nan

    return s.apply(parse_suffix)

# Pronađi sve CSV fajlove
csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_Data.csv")))
print(f"Pronađeno {len(csv_files)} CSV fajlova:\n")

# Kolone koje očekujemo iz izvornih fajlova (Investing.com format)
EXPECTED_COLS = {
    "Date": "Date",
    "Price": "Price",
    "Open": "Open",
    "High": "High",
    "Low": "Low",
    "Change %": "Change %",
}

all_data = []
missing_report = []

for path in csv_files:
    pair_name = os.path.basename(path).replace("_Data.csv", "")
    df_raw = load_csv(path)

    # Procesi datum
    df_raw["Date"] = pd.to_datetime(df_raw["Date"], format="%m/%d/%Y", errors="coerce")

    # Procesi numeričke kolone koje postoje u fajlu
    numeric_cols = [c for c in ["Price", "Open", "High", "Low", "Change %"] if c in df_raw.columns]
    for col in numeric_cols:
        df_raw[col] = clean_numeric_column(df_raw[col])

    # --- Izveštaj o nedostajućim vrednostima PRE dropna() ---
    total_rows = len(df_raw)
    cols_to_check = ["Date"] + numeric_cols
    missing_counts = df_raw[cols_to_check].isna().sum()
    missing_pct = (missing_counts / total_rows * 100).round(2)

    report_row = {"Pair": pair_name, "Total_Rows": total_rows}
    for col in cols_to_check:
        report_row[f"{col}_missing_%"] = missing_pct[col]
    missing_report.append(report_row)

    # Zadrži sve relevantne kolone (Date, Price, Open, High, Low, Vol.)
    keep_cols = ["Date"] + numeric_cols
    df = df_raw[keep_cols].dropna(subset=["Date", "Price"])  # Date i Price su obavezni
    df["Pair"] = pair_name

    all_data.append(df)
    print(f"  ✓ {pair_name:10s}: {len(df)} redova (od {total_rows} ukupno)")

# Kombiniraj SVE parove u JEDAN fajl, kao redove (long format, analogno akcijama)
currency_all = pd.concat(all_data, ignore_index=True)
currency_all = currency_all.sort_values(["Pair", "Date"]).reset_index(drop=True)

print(f"\n✓ Ukupno: {len(currency_all)} redova za {currency_all['Pair'].nunique()} parova")
print(f"✓ Kolone u currency_all: {list(currency_all.columns)}")

Pronađeno 6 CSV fajlova:

  ✓ EURUSD    : 2610 redova (od 2610 ukupno)
  ✓ USDCAD    : 2610 redova (od 2610 ukupno)
  ✓ USDCHF    : 2610 redova (od 2610 ukupno)
  ✓ USDCNY    : 2611 redova (od 2611 ukupno)
  ✓ USDGBP    : 2610 redova (od 2610 ukupno)
  ✓ USDJPY    : 2610 redova (od 2610 ukupno)

✓ Ukupno: 15661 redova za 6 parova
✓ Kolone u currency_all: ['Date', 'Price', 'Open', 'High', 'Low', 'Change %', 'Pair']


### 📊 Izveštaj o nedostajućim vrednostima po valutnom paru

In [35]:
missing_df = pd.DataFrame(missing_report)
print("="*80)
print("UDEO NEDOSTAJUĆIH VREDNOSTI PO PARU (%)")
print("="*80 + "\n")
print(missing_df.to_string(index=False))

missing_df.to_excel('00_Missing_Values_Report.xlsx', index=False)
print(f"\n✓ Izveštaj sačuvan u: 00_Missing_Values_Report.xlsx")

UDEO NEDOSTAJUĆIH VREDNOSTI PO PARU (%)

  Pair  Total_Rows  Date_missing_%  Price_missing_%  Open_missing_%  High_missing_%  Low_missing_%  Change %_missing_%
EURUSD        2610             0.0              0.0             0.0             0.0            0.0                 0.0
USDCAD        2610             0.0              0.0             0.0             0.0            0.0                 0.0
USDCHF        2610             0.0              0.0             0.0             0.0            0.0                 0.0
USDCNY        2611             0.0              0.0             0.0             0.0            0.0                 0.0
USDGBP        2610             0.0              0.0             0.0             0.0            0.0                 0.0
USDJPY        2610             0.0              0.0             0.0             0.0            0.0                 0.0

✓ Izveštaj sačuvan u: 00_Missing_Values_Report.xlsx


## 🔧 FAZA 2+3: Inženjering indikatora (KORIŠĆENJEM `ta` PAKETA)

Umesto ručno pisanih formula, koristimo gotove, testirane funkcije iz `ta` paketa
(`pip install ta`). Pošto imamo i High/Low/Open kolone, koristimo i ATR koji
zahteva ta tri niza, ne samo Price.

In [36]:
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.momentum import ROCIndicator, RSIIndicator
from ta.volatility import AverageTrueRange, BollingerBands

def calculate_indicators(pair_df):
    """
    Izračunaj indikatore koristeći GOTOVE funkcije iz `ta` paketa.
    pair_df mora imati kolone: Price, High, Low (Open opciono) i biti
    sortiran po datumu, za jedan valutni par.
    """
    close = pair_df["Price"]
    high = pair_df["High"] if "High" in pair_df.columns else close
    low = pair_df["Low"] if "Low" in pair_df.columns else close

    out = pd.DataFrame(index=pair_df.index)

    # SMA / EMA (trend)
    out["SMA_14"] = SMAIndicator(close=close, window=14).sma_indicator()
    out["EMA_14"] = EMAIndicator(close=close, window=14).ema_indicator()

    # ROC (momentum)
    out["ROC_14"] = ROCIndicator(close=close, window=14).roc()

    # RSI (momentum)
    out["RSI_14"] = RSIIndicator(close=close, window=14).rsi()

    # MACD (trend)
    macd = MACD(close=close, window_slow=26, window_fast=12, window_sign=9)
    out["MACD"] = macd.macd()
    out["MACD_Signal"] = macd.macd_signal()
    out["MACD_Hist"] = macd.macd_diff()

    # Standardna devijacija cene (preko Bollinger Bands std komponente)
    bb = BollingerBands(close=close, window=14, window_dev=1)
    out["STD_14"] = bb.bollinger_wband() * close / (4 * 100)  # širina trake -> aprox std
    # Napomena: za egzaktan std koristimo direktno rolling, pošto `ta` ne
    # eksponira "sirovi" std kao samostalan indikator
    out["STD_14"] = close.rolling(14).std()

    # ATR (volatilnost) - koristi High/Low/Close, dostupno jer imamo te kolone
    out["ATR_14"] = AverageTrueRange(high=high, low=low, close=close, window=14).average_true_range()

    return out

# Primijeni na sve serije
print("Računanje indikatora pomoću `ta` paketa...\n")

all_features = []
for pair_name in currency_all["Pair"].unique():
    pair_data = currency_all[currency_all["Pair"] == pair_name].sort_values("Date").reset_index(drop=True)

    indicators = calculate_indicators(pair_data)
    indicators["Date"] = pair_data["Date"].values
    indicators["Price"] = pair_data["Price"].values
    indicators["Pair"] = pair_name

    all_features.append(indicators)
    print(f"  ✓ {pair_name}: {len(indicators)} redova sa {indicators.shape[1]-3} indikatora")

features = pd.concat(all_features, ignore_index=True)
features = features[["Pair", "Date", "Price", "SMA_14", "EMA_14", "ROC_14", "RSI_14",
                      "MACD", "MACD_Signal", "MACD_Hist", "STD_14", "ATR_14"]]

print(f"\n✓ Shape: {features.shape}")
print(features.head())

  ✓ USDGBP: 2610 redova sa 9 indikatora
  ✓ USDJPY: 2610 redova sa 9 indikatora

✓ Shape: (15661, 12)
     Pair       Date   Price  SMA_14  EMA_14  ROC_14  RSI_14  MACD  \
0  EURUSD 2016-01-01  1.0858     NaN     NaN     NaN     NaN   NaN   
1  EURUSD 2016-01-04  1.0829     NaN     NaN     NaN     NaN   NaN   
2  EURUSD 2016-01-05  1.0746     NaN     NaN     NaN     NaN   NaN   
3  EURUSD 2016-01-06  1.0778     NaN     NaN     NaN     NaN   NaN   
4  EURUSD 2016-01-07  1.0934     NaN     NaN     NaN     NaN   NaN   

   MACD_Signal  MACD_Hist  STD_14  ATR_14  
0          NaN        NaN     NaN     0.0  
1          NaN        NaN     NaN     0.0  
2          NaN        NaN     NaN     0.0  
3          NaN        NaN     NaN     0.0  
4          NaN        NaN     NaN     0.0  


## 🧠 FAZA 5+6: Neuronska Mreža

In [37]:
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score
)

print("✓ Importovani ML alati")

# Konfiguracija
N_LAGS = 5
TEST_SIZE = 0.2

MLP_PARAMS = {
    'hidden_layer_sizes': (32, 16),
    'activation': 'relu',
    'solver': 'adam',
    'max_iter': 300,
    'early_stopping': True,
    'random_state': 42,
    'verbose': 0
}

print(f"MLP Parametri: {MLP_PARAMS}")

✓ Importovani ML alati
MLP Parametri: {'hidden_layer_sizes': (32, 16), 'activation': 'relu', 'solver': 'adam', 'max_iter': 300, 'early_stopping': True, 'random_state': 42, 'verbose': 0}


In [38]:
def add_lags_and_targets(df, n_lags=5, horizon=1):
    """Dodaj lagged features i target varijable"""
    df_processed = df.copy()

    # Dodaj lagged prices
    for lag in range(1, n_lags + 1):
        df_processed[f'Price_lag{lag}'] = df_processed['Price'].shift(lag)

    # Dodaj target (cijena u budućnosti)
    df_processed['Target_Price'] = df_processed['Price'].shift(-horizon)

    # Dodaj target za klasifikaciju (da li će porast ili pad)
    df_processed['Target_Return'] = df_processed['Target_Price'] - df_processed['Price']
    df_processed['Target_Sign'] = (df_processed['Target_Return'] > 0).astype(int)

    # Uklanja NaN
    df_processed = df_processed.dropna()

    return df_processed

print("✓ Funkcije za feature engineering definisane")

✓ Funkcije za feature engineering definisane


In [39]:
# Pripremi podatke za eksperimente
results_list = []

for pair_name in features['Pair'].unique():
    print(f"\n{'='*60}")
    print(f"OBRADA: {pair_name}")
    print(f"{'='*60}")

    pair_features = features[features['Pair'] == pair_name].copy()
    pair_features = pair_features.sort_values('Date')

    # Dodaj features
    pair_processed = add_lags_and_targets(pair_features, n_lags=N_LAGS, horizon=1)

    if len(pair_processed) < 100:
        print(f"⚠️  Premalo podataka ({len(pair_processed)})")
        continue

    # Definisanja features
    feature_cols = ['SMA_14', 'EMA_14', 'ROC_14', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist', 'STD_14', 'ATR_14']
    feature_cols += [f'Price_lag{i}' for i in range(1, N_LAGS + 1)]
    feature_cols = [c for c in feature_cols if c in pair_processed.columns]

    # Podijeli na train/test
    split_idx = int(len(pair_processed) * (1 - TEST_SIZE))

    X_train = pair_processed.iloc[:split_idx][feature_cols].values
    X_test = pair_processed.iloc[split_idx:][feature_cols].values

    y_train_reg = pair_processed.iloc[:split_idx]['Target_Price'].values
    y_test_reg = pair_processed.iloc[split_idx:]['Target_Price'].values

    y_train_cls = pair_processed.iloc[:split_idx]['Target_Sign'].values
    y_test_cls = pair_processed.iloc[split_idx:]['Target_Sign'].values

    print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")
    print(f"Features: {len(feature_cols)}")

    # Normalizeuj
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # REGRESIJA - Predikcija cijene
    print("\n📈 REGRESIJA (Predikcija cijene)...")
    try:
        model_reg = MLPRegressor(**MLP_PARAMS)
        model_reg.fit(X_train_scaled, y_train_reg)
        y_pred_reg = model_reg.predict(X_test_scaled)

        mae = mean_absolute_error(y_test_reg, y_pred_reg)
        rmse = sqrt(mean_squared_error(y_test_reg, y_pred_reg))
        r2 = r2_score(y_test_reg, y_pred_reg)

        print(f"  MAE: {mae:.6f}")
        print(f"  RMSE: {rmse:.6f}")
        print(f"  R²: {r2:.4f}")

        results_list.append({
            'Pair': pair_name,
            'Model': 'Regresija',
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2,
            'Train_Size': len(X_train),
            'Test_Size': len(X_test)
        })
    except Exception as e:
        print(f"  ❌ Greška: {str(e)[:50]}")

    # KLASIFIKACIJA - Predviđanje smjera (gore/dolje)
    print("\n📊 KLASIFIKACIJA (Predviđanje smjera)...")
    try:
        # Provjeri ako imamo obe klase
        if len(np.unique(y_train_cls)) < 2 or len(np.unique(y_test_cls)) < 2:
            print("  ⚠️  Nema dovoljno klasa za obuku")
        else:
            model_cls = MLPClassifier(**MLP_PARAMS)
            model_cls.fit(X_train_scaled, y_train_cls)
            y_pred_cls = model_cls.predict(X_test_scaled)

            acc = accuracy_score(y_test_cls, y_pred_cls)
            prec = precision_score(y_test_cls, y_pred_cls, zero_division=0)
            rec = recall_score(y_test_cls, y_pred_cls, zero_division=0)
            f1 = f1_score(y_test_cls, y_pred_cls, zero_division=0)

            print(f"  Accuracy: {acc:.4f}")
            print(f"  Precision: {prec:.4f}")
            print(f"  Recall: {rec:.4f}")
            print(f"  F1: {f1:.4f}")

            results_list.append({
                'Pair': pair_name,
                'Model': 'Klasifikacija',
                'Accuracy': acc,
                'Precision': prec,
                'Recall': rec,
                'F1': f1,
                'Train_Size': len(X_train),
                'Test_Size': len(X_test)
            })
    except Exception as e:
        print(f"  ❌ Greška: {str(e)[:50]}")

print(f"\n{'='*60}")
print(f"✓ Eksperimenti završeni!")
print(f"{'='*60}")

  MAE: 9.180209
  RMSE: 10.126530
  R²: -3.4398

📊 KLASIFIKACIJA (Predviđanje smjera)...
  Accuracy: 0.5407
  Precision: 0.5419
  Recall: 0.8945
  F1: 0.6749

✓ Eksperimenti završeni!


## 📊 Analiza rezultata

In [40]:
# Kreiraj DataFrame sa rezultatima
if results_list:
    results_df = pd.DataFrame(results_list)
    print("\n" + "="*80)
    print("REZULTATI")
    print("="*80 + "\n")
    print(results_df.to_string(index=False))

    # Čuva u Excel
    results_df.to_excel('NN_Results.xlsx', index=False)
    print(f"\n✓ Rezultati sačuvani u: NN_Results.xlsx")
else:
    print("❌ Nema rezultata")


REZULTATI

  Pair         Model      MAE      RMSE        R2  Train_Size  Test_Size  Accuracy  Precision   Recall       F1
EURUSD     Regresija 0.021593  0.030744  0.522020        2060        516       NaN        NaN      NaN      NaN
EURUSD Klasifikacija      NaN       NaN       NaN        2060        516  0.523256   0.502577 0.786290 0.613208
USDCAD     Regresija 0.031521  0.041346 -1.283222        2060        516       NaN        NaN      NaN      NaN
USDCAD Klasifikacija      NaN       NaN       NaN        2060        516  0.480620   0.534483 0.113971 0.187879
USDCHF     Regresija 0.038869  0.056627 -0.833302        2060        516       NaN        NaN      NaN      NaN
USDCHF Klasifikacija      NaN       NaN       NaN        2060        516  0.513566   0.545455 0.273764 0.364557
USDCNY     Regresija 0.084312  0.107889 -1.038499        2061        516       NaN        NaN      NaN      NaN
USDCNY Klasifikacija      NaN       NaN       NaN        2061        516  0.498062   0.49541

## 💾 Čuvanje značajnih fajlova

In [41]:
# Čuva osnove podatke
currency_all.to_csv('01_Currency_Data_ALL.csv', index=False)
features.to_csv('02_Features_With_Indicators.csv', index=False)

print("✓ Sačuvani fajlovi:")
print("  - 00_Missing_Values_Report.xlsx")
print("  - 01_Currency_Data_ALL.csv")
print("  - 02_Features_With_Indicators.csv")
print("  - NN_Results.xlsx")

✓ Sačuvani fajlovi:
  - 00_Missing_Values_Report.xlsx
  - 01_Currency_Data_ALL.csv
  - 02_Features_With_Indicators.csv
  - NN_Results.xlsx
